In [ ]:
%%sql -r dataframe_1
---SELECT DATABASE

USE DATABASE EYPROJECT;
USE SCHEMA PUBLIC;

# EY Data & AI Challenge – Water Quality Prediction

## Objective
Predict three South African river water-quality parameters on the validation dataset:
- Total Alkalinity
- Electrical Conductance
- Dissolved Reactive Phosphorus

## Evaluation
Leaderboard metric: mean R² across the three targets.

## Modelling strategy
This notebook builds an ensemble of XGBoost and LightGBM models using:
- Landsat features
- TerraClimate features
- Terrain and coastal features
- Soil features
- Temporal and interaction features
- Optional spatial nearest-neighbour features

## Important note
The validation region comes from different regions not present in training, so spatial generalization is critical.

In [ ]:
#Installing python packages from requirement text file provided by EY
!pip install uv
!uv pip install  -r requirements.txt 
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

!pip install lightgbm
from lightgbm import LGBMRegressor

Step 2: Reading csv files


In [ ]:
# Load datasets
water_df = pd.read_csv("water_quality_training_dataset.csv")
landsat_df = pd.read_csv("landsat_features_training.csv")
terraclimate_df = pd.read_csv("terraclimate_features_training.csv")

# Standardize keys
for df in [water_df, landsat_df, terraclimate_df]:
    df["Sample Date"] = pd.to_datetime(df["Sample Date"], dayfirst=True, format="mixed", errors="coerce").dt.date
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce").round(4)
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce").round(4)

# Clean Landsat numeric columns
landsat_cols = [c for c in landsat_df.columns if c not in ["Longitude", "Latitude", "Sample Date"]]
for col in landsat_cols:
    landsat_df[col] = pd.to_numeric(landsat_df[col], errors="coerce")

# Clean TerraClimate numeric columns
terraclimate_cols = [c for c in terraclimate_df.columns if c not in ["Longitude", "Latitude", "Sample Date"]]
for col in terraclimate_cols:
    terraclimate_df[col] = pd.to_numeric(terraclimate_df[col], errors="coerce")

# Drop duplicate feature keys before merging
landsat_df = landsat_df.drop_duplicates(subset=["Longitude", "Latitude", "Sample Date"])
terraclimate_df = terraclimate_df.drop_duplicates(subset=["Longitude", "Latitude", "Sample Date"])

# Merge
data = water_df.merge(
    landsat_df,
    on=["Longitude", "Latitude", "Sample Date"],
    how="left"
).merge(
    terraclimate_df,
    on=["Longitude", "Latitude", "Sample Date"],
    how="left"
)

# Add missingness flags for Landsat (important signal for tree models)
for col in landsat_cols:
    data[f"{col}_missing"] = data[col].isna().astype(int)

# Fill Landsat missing values with training medians
for col in landsat_cols:
    data[col] = data[col].fillna(data[col].median())

# Fill TerraClimate missing values with training medians
for col in terraclimate_cols:
    data[col] = data[col].fillna(data[col].median())

# Final checks
print("Rows in water_df:", len(water_df))
print("Rows after merge:", len(data))
print("Duplicate keys in final data:", data.duplicated(subset=["Longitude", "Latitude", "Sample Date"]).sum())
print("\nRemaining missing values (top 20):")
print(data.isna().mean().sort_values(ascending=False).head(20))

In [ ]:
# Download file from user stage to local /tmp directory
session.sql("""
    GET @~/terraclimate_ppt_training_mapped.csv
    file:///tmp/
""").collect()
session.sql("""
    GET @~/terraclimate_ppt_validation_mapped.csv
    file:///tmp/
""").collect()

print("Downloaded from stage to /tmp/")

In [ ]:
# Load both PPT files
ppt_train = pd.read_csv("/tmp/terraclimate_ppt_training_mapped.csv")
ppt_val = pd.read_csv("/tmp/terraclimate_ppt_validation_mapped.csv")

# Standardize dates
ppt_train["Sample Date"] = pd.to_datetime(ppt_train["Sample Date"], dayfirst=True, format="mixed", errors="coerce").dt.normalize()
ppt_val["Sample Date"] = pd.to_datetime(ppt_val["Sample Date"], dayfirst=True, format="mixed", errors="coerce").dt.normalize()

# Standardize coords
for df in [ppt_train, ppt_val]:
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce").round(4)
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce").round(4)

display(ppt_train.head())
display(ppt_val.head())

print("ppt_train shape:", ppt_train.shape)
print("ppt_val shape:", ppt_val.shape)
print("ppt_train columns:", ppt_train.columns.tolist())
print("ppt_val columns:", ppt_val.columns.tolist())

In [ ]:
data["Sample Date"] = pd.to_datetime(data["Sample Date"], dayfirst=True)

data = data.merge(
    ppt_train,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

print("After merge shape:", data.shape)
print("Missing ppt:", data["ppt"].isna().sum())

In [ ]:
session.sql("""
    GET @~/terraclimate_tmax_training_mapped.csv
    file:///tmp/
""").collect()

session.sql("""
    GET @~/terraclimate_tmax_validation_mapped.csv
    file:///tmp/
""").collect()

print("Downloaded tmax files from stage to /tmp/")

In [ ]:
tmax_train = pd.read_csv("/tmp/terraclimate_tmax_training_mapped.csv")
tmax_val = pd.read_csv("/tmp/terraclimate_tmax_validation_mapped.csv")

tmax_train["Sample Date"] = pd.to_datetime(tmax_train["Sample Date"], dayfirst=True)
tmax_val["Sample Date"] = pd.to_datetime(tmax_val["Sample Date"], dayfirst=True)

display(tmax_train.head())
display(tmax_val.head())

print("tmax_train shape:", tmax_train.shape)
print("tmax_val shape:", tmax_val.shape)
print("tmax_train columns:", tmax_train.columns.tolist())
print("tmax_val columns:", tmax_val.columns.tolist())

In [ ]:
data = data.merge(
    tmax_train,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

print("After tmax merge:", data.shape)
print("Missing tmax rows:", data["tmax"].isna().sum())
print("Duplicate Lat/Lon/Date rows after tmax merge:", data.duplicated(subset=["Latitude", "Longitude", "Sample Date"]).sum())

In [ ]:
session.sql("""
    GET @~/terraclimate_tmin_training_mapped.csv
    file:///tmp/
""").collect()

session.sql("""
    GET @~/terraclimate_tmin_validation_mapped.csv
    file:///tmp/
""").collect()

print("Downloaded tmin files from stage to /tmp/")

In [ ]:
tmin_train = pd.read_csv("/tmp/terraclimate_tmin_training_mapped.csv")
tmin_val = pd.read_csv("/tmp/terraclimate_tmin_validation_mapped.csv")

tmin_train["Sample Date"] = pd.to_datetime(tmin_train["Sample Date"], dayfirst=True)
tmin_val["Sample Date"] = pd.to_datetime(tmin_val["Sample Date"], dayfirst=True)

display(tmin_train.head())
display(tmin_val.head())

print("tmin_train shape:", tmin_train.shape)
print("tmin_val shape:", tmin_val.shape)
print("tmin_train columns:", tmin_train.columns.tolist())
print("tmin_val columns:", tmin_val.columns.tolist())

In [ ]:
data = data.merge(
    tmin_train,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

print("After tmin merge:", data.shape)
print("Missing tmin rows:", data["tmin"].isna().sum())
print("Duplicate Lat/Lon/Date rows after tmin merge:", data.duplicated(subset=["Latitude", "Longitude", "Sample Date"]).sum())

In [ ]:
session.sql("""
    GET @~/elevation_features_training.csv
    file:///tmp/
""").collect()

session.sql("""
    GET @~/elevation_features_validation.csv
    file:///tmp/
""").collect()

print("Downloaded elevation files from stage to /tmp/")

In [ ]:
elev_train = pd.read_csv("/tmp/elevation_features_training.csv")
elev_val = pd.read_csv("/tmp/elevation_features_validation.csv")

elev_train["Sample Date"] = pd.to_datetime(elev_train["Sample Date"], format="mixed", errors="coerce").dt.normalize()
elev_val["Sample Date"] = pd.to_datetime(elev_val["Sample Date"], format="mixed", errors="coerce").dt.normalize()

print("Missing parsed train dates:", elev_train["Sample Date"].isna().sum())
print("Missing parsed val dates:", elev_val["Sample Date"].isna().sum())
print("elev_train shape:", elev_train.shape)
print("elev_val shape:", elev_val.shape)
print("elev_train columns:", elev_train.columns.tolist())
print("elev_val columns:", elev_val.columns.tolist())
display(elev_train.head())
display(elev_val.head())

In [ ]:
data = data.merge(
    elev_train,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

print("After elevation merge:", data.shape)
print("Missing elevation rows:", data["elevation"].isna().sum())
print(
    "Duplicate Lat/Lon/Date rows after elevation merge:",
    data.duplicated(subset=["Latitude", "Longitude", "Sample Date"]).sum()
)

In [ ]:
session.sql("""
    GET @~/landcover_training.csv
    file:///tmp/
""").collect()

session.sql("""
    GET @~/landcover_validation.csv
    file:///tmp/
""").collect()

print("Downloaded landcover files from stage to /tmp/")

In [ ]:
train_lc = pd.read_csv("/tmp/landcover_training.csv")
val_lc = pd.read_csv("/tmp/landcover_validation.csv")

train_lc["Sample Date"] = pd.to_datetime(train_lc["Sample Date"], format="mixed", errors="coerce").dt.normalize()
val_lc["Sample Date"] = pd.to_datetime(val_lc["Sample Date"], format="mixed", errors="coerce").dt.normalize()

In [ ]:
train_lc.head(5)

In [ ]:
train_lc = train_lc.copy()

if "Sample Date" in train_lc.columns:
    train_lc = train_lc.drop(columns=["Sample Date"])

train_lc["Latitude"] = pd.to_numeric(train_lc["Latitude"], errors="coerce").round(4)
train_lc["Longitude"] = pd.to_numeric(train_lc["Longitude"], errors="coerce").round(4)

data["Latitude"] = pd.to_numeric(data["Latitude"], errors="coerce").round(4)
data["Longitude"] = pd.to_numeric(data["Longitude"], errors="coerce").round(4)

train_lc = train_lc.drop_duplicates(subset=["Latitude", "Longitude"])

data = data.merge(
    train_lc,
    on=["Latitude", "Longitude"],
    how="left"
)

print("After landcover merge:", data.shape)
print("Missing landcover rows:", data["landcover"].isna().sum())
print(
    "Duplicate Lat/Lon/Date rows after landcover merge:",
    data.duplicated(subset=["Latitude", "Longitude", "Sample Date"]).sum()
)

In [ ]:
session.sql("""
    GET @~/coast_distance_training.csv
    file:///tmp/
""").collect()

session.sql("""
    GET @~/coast_distance_validation.csv
    file:///tmp/
""").collect()

print("Downloaded coast distance files from stage to /tmp/")

In [ ]:
train_coast = pd.read_csv("/tmp/coast_distance_training.csv")
val_coast = pd.read_csv("/tmp/coast_distance_validation.csv")

train_coast["Sample Date"] = pd.to_datetime(train_coast["Sample Date"], format="mixed", errors="coerce").dt.normalize()
val_coast["Sample Date"] = pd.to_datetime(val_coast["Sample Date"], format="mixed", errors="coerce").dt.normalize()

In [ ]:
print("rows in train_terrain:", len(train_terrain))
print("unique coords in train_terrain:", train_terrain[["Latitude","Longitude"]].drop_duplicates().shape)

In [ ]:
train_coast.head(5)

In [ ]:
train_coast = train_coast.copy()

# coast distance should be treated as static
if "Sample Date" in train_coast.columns:
    train_coast = train_coast.drop(columns=["Sample Date"])

# clean column names
train_coast.columns = train_coast.columns.str.strip()
data.columns = data.columns.str.strip()

# clean keys
train_coast["Latitude"] = pd.to_numeric(train_coast["Latitude"], errors="coerce").round(4)
train_coast["Longitude"] = pd.to_numeric(train_coast["Longitude"], errors="coerce").round(4)

data["Latitude"] = pd.to_numeric(data["Latitude"], errors="coerce").round(4)
data["Longitude"] = pd.to_numeric(data["Longitude"], errors="coerce").round(4)

# drop any old coast-distance columns already in data
coast_cols_to_drop = [c for c in data.columns if "coast" in c.lower()]
if coast_cols_to_drop:
    data = data.drop(columns=coast_cols_to_drop)

# keep one row per coordinate in coast table
train_coast = train_coast.drop_duplicates(subset=["Latitude", "Longitude"])

# merge on coordinates only
data = data.merge(
    train_coast,
    on=["Latitude", "Longitude"],
    how="left"
)

print("After coast distance merge:", data.shape)
print("Missing coast distance rows:", data["dist_coast_km"].isna().sum())
print(
    "Duplicate Lat/Lon/Date rows after coast distance merge:",
    data.duplicated(subset=["Latitude", "Longitude", "Sample Date"]).sum()
)
# Handle coast distance feature
if "dist_coast_km" in data.columns:
    
    # Fill missing values
    data["dist_coast_km"] = data["dist_coast_km"].fillna(data["dist_coast_km"].median())

    # Create log-transformed feature
    data["dist_coast_log"] = np.log1p(data["dist_coast_km"])

    print("Missing coast distances after fill:", data["dist_coast_km"].isna().sum())
    print(data[["dist_coast_km","dist_coast_log"]].describe())

In [ ]:
session.sql("""
    GET @~/terrain_features_training.csv
    file:///tmp/
""").collect()

session.sql("""
    GET @~/terrain_features_validation.csv
    file:///tmp/
""").collect()

print("Downloaded terrain features files from stage to /tmp/")

In [ ]:
train_terrain = pd.read_csv("/tmp/terrain_features_training.csv")
val_terrain = pd.read_csv("/tmp/terrain_features_validation.csv")

train_terrain["Sample Date"] = pd.to_datetime(
    train_terrain["Sample Date"],
    format="mixed",
    errors="coerce"
).dt.normalize()

val_terrain["Sample Date"] = pd.to_datetime(
    val_terrain["Sample Date"],
    format="mixed",
    errors="coerce"
).dt.normalize()

In [ ]:
train_terrain

In [ ]:
train_terrain.head(5)
print(train_terrain[["elevation", "slope_deg", "local_relief", "roughness"]].isna().sum())

In [ ]:
train_terrain = train_terrain.copy()

# clean column names first
train_terrain.columns = train_terrain.columns.str.strip()
data.columns = data.columns.str.strip()

# terrain is static -> do not merge on Sample Date
if "Sample Date" in train_terrain.columns:
    train_terrain = train_terrain.drop(columns=["Sample Date"])

# clean keys
train_terrain["Latitude"] = pd.to_numeric(train_terrain["Latitude"], errors="coerce").round(4)
train_terrain["Longitude"] = pd.to_numeric(train_terrain["Longitude"], errors="coerce").round(4)

data["Latitude"] = pd.to_numeric(data["Latitude"], errors="coerce").round(4)
data["Longitude"] = pd.to_numeric(data["Longitude"], errors="coerce").round(4)

# keep one row per coordinate
train_terrain = train_terrain.drop_duplicates(subset=["Latitude", "Longitude"])

# drop old terrain columns if already present
terrain_cols_to_drop = [c for c in ["elevation", "slope_deg", "local_relief", "roughness"] if c in data.columns]
if terrain_cols_to_drop:
    data = data.drop(columns=terrain_cols_to_drop)

# merge on coordinates only
data = data.merge(
    train_terrain,
    on=["Latitude", "Longitude"],
    how="left"
)

print("After terrain merge:", data.shape)
print("Missing terrain rows:")
print(data[["elevation", "slope_deg", "local_relief", "roughness"]].isna().sum())
print(
    "Duplicate Lat/Lon/Date rows after terrain merge:",
    data.duplicated(subset=["Latitude", "Longitude", "Sample Date"]).sum()
)

In [ ]:
# LOAD SOIL FEATURES

soil_train = pd.read_csv("soil_training.csv")

soil_val = pd.read_csv("soil_validation.csv")

print("Soil train shape:", soil_train.shape)
print("Soil validation shape:", soil_val.shape)

display(soil_train.head())

soil_train["Sample Date"] = pd.to_datetime(
    soil_train["Sample Date"],
    format="mixed",
    errors="coerce"
).dt.normalize()

soil_val["Sample Date"] = pd.to_datetime(
    soil_val["Sample Date"],
    format="mixed",
    errors="coerce"
).dt.normalize()

In [ ]:
soil_train

In [ ]:
# ===== ENHANCED SOIL CHECK + MERGE + FILL =====

soil_train = soil_train.copy()
data = data.copy()

# Clean column names
soil_train.columns = soil_train.columns.str.strip()
data.columns = data.columns.str.strip()

# Standardize keys BEFORE merge
for df in [soil_train, data]:
    if "Sample Date" in df.columns:
        df["Sample Date"] = pd.to_datetime(
            df["Sample Date"],
            dayfirst=True,
            format="mixed",
            errors="coerce"
        ).dt.normalize()

    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce").round(4)
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce").round(4)

# Soil is static -> do not merge on Sample Date
if "Sample Date" in soil_train.columns:
    soil_train = soil_train.drop(columns=["Sample Date"])

# Soil feature columns
soil_cols = [
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_carbon",
    "soil_bulk_density",
    "soil_ph",
    "soil_cec"
]

# Keep only soil columns that actually exist
soil_cols = [c for c in soil_cols if c in soil_train.columns]

print("soil_train shape before dedupe:", soil_train.shape)
print("Missing in soil_train before merge:")
print(soil_train[soil_cols].isna().sum())

# Keep one soil row per coordinate
soil_train = soil_train.drop_duplicates(subset=["Latitude", "Longitude"])

print("\nsoil_train shape after dedupe:", soil_train.shape)

# Drop any old soil columns / flags already in data
cols_to_drop = [c for c in data.columns if c in soil_cols or c.endswith("_missing") and c.replace("_missing", "") in soil_cols]
if cols_to_drop:
    data = data.drop(columns=cols_to_drop)

rows_before = len(data)

# Merge on coordinates only
data = data.merge(
    soil_train[["Latitude", "Longitude"] + soil_cols],
    on=["Latitude", "Longitude"],
    how="left"
)

rows_after = len(data)

print("\nAfter soil merge:", data.shape)
print("Rows before merge:", rows_before)
print("Rows after merge:", rows_after)

print("\nMissing soil rows after merge:")
print(data[soil_cols].isna().sum())

print(
    "Duplicate Lat/Lon/Date rows after soil merge:",
    data.duplicated(subset=["Latitude", "Longitude", "Sample Date"]).sum()
)

# Create missingness flags BEFORE fill
for col in soil_cols:
    data[f"{col}_missing"] = data[col].isna().astype(int)

# Median fill
for col in soil_cols:
    data[col] = data[col].fillna(data[col].median())

print("\nMissing soil rows after median fill:")
print(data[soil_cols].isna().sum())

print("\nCreated missing flags:")
print([f"{c}_missing" for c in soil_cols])

display(data[["Latitude", "Longitude", "Sample Date"] + soil_cols + [f"{c}_missing" for c in soil_cols]].head())

In [ ]:
def standardize_keys(df):
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "Sample Date" in df.columns:
        df["Sample Date"] = pd.to_datetime(
            df["Sample Date"],
            dayfirst=True,
            format="mixed",
            errors="coerce"
        ).dt.normalize()

    if "Latitude" in df.columns:
        df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce").round(4)

    if "Longitude" in df.columns:
        df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce").round(4)

    return df

In [ ]:
water_df = standardize_keys(water_df)
landsat_df = standardize_keys(landsat_df)
terraclimate_df = standardize_keys(terraclimate_df)
ppt_train = standardize_keys(ppt_train)
ppt_val = standardize_keys(ppt_val)
tmax_train = standardize_keys(tmax_train)
tmax_val = standardize_keys(tmax_val)
tmin_train = standardize_keys(tmin_train)
tmin_val = standardize_keys(tmin_val)
elev_train = standardize_keys(elev_train)
elev_val = standardize_keys(elev_val)
soil_train = standardize_keys(soil_train)
soil_val = standardize_keys(soil_val)
train_terrain = standardize_keys(train_terrain)
val_terrain = standardize_keys(val_terrain)
train_lc = standardize_keys(train_lc)
val_lc = standardize_keys(val_lc)
train_coast = standardize_keys(train_coast)
val_coast = standardize_keys(val_coast)

In [ ]:
# ---------- FINAL SAFE TRAIN / VALIDATION MERGE BLOCK ----------

import pandas as pd

# Load validation template if not already loaded
if "val_df" not in globals():
    val_df = pd.read_csv("submission_template.csv")

val_df = standardize_keys(val_df)

# STATIC datasets must merge on Latitude + Longitude only
STATIC_FEATURES = {
    "soil_train", "soil_val",
    "train_terrain", "val_terrain",
    "train_lc", "val_lc",
    "train_coast", "val_coast",
    "elev_train", "elev_val"
}

def prep_feature(df, name):
    df = df.copy()
    df.columns = df.columns.str.strip()

    if name in STATIC_FEATURES:
        if "Sample Date" in df.columns:
            df = df.drop(columns=["Sample Date"])
        merge_keys = ["Longitude", "Latitude"]
    else:
        merge_keys = ["Longitude", "Latitude", "Sample Date"]

    before = len(df)
    df = df.drop_duplicates(subset=merge_keys)
    after = len(df)

    print(f"{name}: using merge keys = {merge_keys}")
    print(f"{name}: rows before dedupe = {before}, after dedupe = {after}")

    return df, merge_keys

def safe_merge(base_df, feature_df, name):
    feature_df, merge_keys = prep_feature(feature_df, name)

    base_before = len(base_df)
    out = base_df.merge(feature_df, on=merge_keys, how="left")
    base_after = len(out)

    print(f"{name}: base rows before merge = {base_before}, after merge = {base_after}")

    if base_before != base_after:
        print(f"WARNING: row count changed after merging {name}")

    return out

# Start base tables
train_data = water_df.copy()
validation_data = val_df.copy()

# -------- TRAIN MERGES --------
train_data = safe_merge(train_data, landsat_df, "landsat_df")
train_data = safe_merge(train_data, terraclimate_df, "terraclimate_df")
train_data = safe_merge(train_data, ppt_train, "ppt_train")
train_data = safe_merge(train_data, tmax_train, "tmax_train")
train_data = safe_merge(train_data, tmin_train, "tmin_train")
train_data = safe_merge(train_data, elev_train, "elev_train")
train_data = safe_merge(train_data, soil_train, "soil_train")
train_data = safe_merge(train_data, train_terrain, "train_terrain")
train_data = safe_merge(train_data, train_lc, "train_lc")
train_data = safe_merge(train_data, train_coast, "train_coast")

# -------- VALIDATION MERGES --------
validation_data = safe_merge(validation_data, ppt_val, "ppt_val")
validation_data = safe_merge(validation_data, tmax_val, "tmax_val")
validation_data = safe_merge(validation_data, tmin_val, "tmin_val")
validation_data = safe_merge(validation_data, elev_val, "elev_val")
validation_data = safe_merge(validation_data, soil_val, "soil_val")
validation_data = safe_merge(validation_data, val_terrain, "val_terrain")
validation_data = safe_merge(validation_data, val_lc, "val_lc")
validation_data = safe_merge(validation_data, val_coast, "val_coast")

# Fix duplicate elevation columns
if "elevation_y" in train_data.columns:
    train_data = train_data.drop(columns=["elevation_y"])
if "elevation_y" in validation_data.columns:
    validation_data = validation_data.drop(columns=["elevation_y"])

if "elevation_x" in train_data.columns:
    train_data = train_data.rename(columns={"elevation_x": "elevation"})
if "elevation_x" in validation_data.columns:
    validation_data = validation_data.rename(columns={"elevation_x": "elevation"})

# Display Sample Date cleanly AFTER merges
if "Sample Date" in train_data.columns:
    train_data["Sample Date"] = pd.to_datetime(train_data["Sample Date"], errors="coerce").dt.date
if "Sample Date" in validation_data.columns:
    validation_data["Sample Date"] = pd.to_datetime(validation_data["Sample Date"], errors="coerce").dt.date

print("\nFINAL SHAPES")
print("train_data shape:", train_data.shape)
print("validation_data shape:", validation_data.shape)

print("\nTop missingness in train_data:")
print(train_data.isna().mean().sort_values(ascending=False).head(30))

print("\nTop missingness in validation_data:")
print(validation_data.isna().mean().sort_values(ascending=False).head(30))

display(train_data.head())
display(validation_data.head())

In [ ]:
# ===== FINAL FEATURE ENGINEERING: TRAIN + VALIDATION =====

import numpy as np
import pandas as pd

# Work on copies
train_data = train_data.copy()
validation_data = validation_data.copy()

# -------------------------
# Clean column names once
# -------------------------
train_data.columns = train_data.columns.str.strip().str.lower().str.replace(" ", "_")
validation_data.columns = validation_data.columns.str.strip().str.lower().str.replace(" ", "_")

# -------------------------
# Date handling
# -------------------------
for df in [train_data, validation_data]:
    df["sample_date"] = pd.to_datetime(df["sample_date"], errors="coerce")
    df["year"] = df["sample_date"].dt.year
    df["month"] = df["sample_date"].dt.month
    df["dayofyear"] = df["sample_date"].dt.dayofyear
    df["season"] = ((df["month"] % 12 + 3) // 3)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["wet_season"] = df["month"].isin([10, 11, 12, 1, 2, 3]).astype(int)

# location id for grouped CV
train_data["location_id"] = (
    train_data["longitude"].round(5).astype(str) + "_" +
    train_data["latitude"].round(5).astype(str)
)

validation_data["location_id"] = (
    validation_data["longitude"].round(5).astype(str) + "_" +
    validation_data["latitude"].round(5).astype(str)
)

# -------------------------
# Landsat
# -------------------------
landsat_cols = ["nir", "green", "swir16", "swir22", "ndmi", "mndwi"]
existing_landsat_cols = [c for c in landsat_cols if c in train_data.columns and c in validation_data.columns]

print("Using Landsat columns:", existing_landsat_cols)

if len(existing_landsat_cols) > 0:
    train_data["landsat_missing"] = train_data[existing_landsat_cols].isna().any(axis=1).astype(int)
    validation_data["landsat_missing"] = validation_data[existing_landsat_cols].isna().any(axis=1).astype(int)

    for c in existing_landsat_cols:
        train_data[c] = pd.to_numeric(train_data[c], errors="coerce")
        validation_data[c] = pd.to_numeric(validation_data[c], errors="coerce")

        med = train_data[c].median()

        train_data[f"{c}_missing"] = train_data[c].isna().astype(int)
        validation_data[f"{c}_missing"] = validation_data[c].isna().astype(int)

        train_data[c] = train_data[c].fillna(med)
        validation_data[c] = validation_data[c].fillna(med)

# -------------------------
# Climate
# -------------------------
climate_cols = ["pet", "ppt", "tmax", "tmin"]

for c in climate_cols:
    if c in train_data.columns and c in validation_data.columns:
        train_data[c] = pd.to_numeric(train_data[c], errors="coerce")
        validation_data[c] = pd.to_numeric(validation_data[c], errors="coerce")

        med = train_data[c].median()
        train_data[c] = train_data[c].fillna(med)
        validation_data[c] = validation_data[c].fillna(med)

for df in [train_data, validation_data]:
    if "pet" in df.columns and "ppt" in df.columns:
        df["dryness_index"] = df["pet"] / (df["ppt"] + 1e-6)
        df["ppt_pet_ratio"] = df["ppt"] / (df["pet"] + 1e-6)
        df["water_balance"] = df["ppt"] - df["pet"]
        df["water_balance_ratio"] = (df["ppt"] - df["pet"]) / (df["ppt"] + df["pet"] + 1e-6)
        df["dryness_x_wetseason"] = df["dryness_index"] * df["wet_season"]
        df["ppt_log"] = np.log1p(np.clip(df["ppt"], 0, None))
        df["pet_log"] = np.log1p(np.clip(df["pet"], 0, None))

    if "tmax" in df.columns and "tmin" in df.columns:
        df["temp_range"] = df["tmax"] - df["tmin"]

# -------------------------
# Terrain / coast
# -------------------------
terrain_cols = ["elevation", "slope_deg", "local_relief", "roughness", "dist_coast_km"]

for c in terrain_cols:
    if c in train_data.columns and c in validation_data.columns:
        train_data[c] = pd.to_numeric(train_data[c], errors="coerce")
        validation_data[c] = pd.to_numeric(validation_data[c], errors="coerce")

        med = train_data[c].median()
        train_data[c] = train_data[c].fillna(med)
        validation_data[c] = validation_data[c].fillna(med)

for df in [train_data, validation_data]:
    if "elevation" in df.columns:
        df["elevation_squared"] = df["elevation"] ** 2
    if "slope_deg" in df.columns:
        df["slope_squared"] = df["slope_deg"] ** 2
    if "dist_coast_km" in df.columns:
        df["dist_coast_log"] = np.log1p(df["dist_coast_km"])

# -------------------------
# Terrain / climate interactions
# -------------------------
for df in [train_data, validation_data]:
    if "elevation" in df.columns and "ppt" in df.columns:
        df["elevation_x_ppt"] = df["elevation"] * df["ppt"]
    if "elevation" in df.columns and "pet" in df.columns:
        df["elevation_x_pet"] = df["elevation"] * df["pet"]
    if "slope_deg" in df.columns and "ppt" in df.columns:
        df["slope_x_ppt"] = df["slope_deg"] * df["ppt"]
    if "local_relief" in df.columns and "pet" in df.columns:
        df["relief_x_pet"] = df["local_relief"] * df["pet"]
    if "dist_coast_km" in df.columns and "ppt" in df.columns:
        df["coast_x_ppt"] = df["dist_coast_km"] * df["ppt"]
    if "dist_coast_km" in df.columns and "pet" in df.columns:
        df["coast_x_pet"] = df["dist_coast_km"] * df["pet"]
    if "dist_coast_km" in df.columns and "dryness_index" in df.columns:
        df["coast_x_dryness"] = df["dist_coast_km"] * df["dryness_index"]
    if "elevation" in df.columns and "dryness_index" in df.columns:
        df["elevation_x_dryness"] = df["elevation"] * df["dryness_index"]
    if "slope_deg" in df.columns and "water_balance" in df.columns:
        df["slope_x_water_balance"] = df["slope_deg"] * df["water_balance"]

# -------------------------
# Lag climate features
# -------------------------
for df in [train_data, validation_data]:
    df.sort_values(["location_id", "sample_date"], inplace=True)
    df.reset_index(drop=True, inplace=True)

    if "ppt" in df.columns:
        df["ppt_lag1"] = df.groupby("location_id")["ppt"].shift(1)
        df["ppt_roll3"] = (
            df.groupby("location_id")["ppt"]
            .rolling(3).mean()
            .reset_index(level=0, drop=True)
        )
        df["ppt_delta"] = df.groupby("location_id")["ppt"].diff()
        df["ppt_max3"] = (
            df.groupby("location_id")["ppt"]
            .rolling(3).max()
            .reset_index(level=0, drop=True)
        )

lag_cols = ["ppt_lag1", "ppt_roll3", "ppt_delta", "ppt_max3"]
for c in lag_cols:
    if c in train_data.columns and c in validation_data.columns:
        med = train_data[c].median()
        train_data[c] = train_data[c].fillna(med)
        validation_data[c] = validation_data[c].fillna(med)

# -------------------------
# Spectral interactions
# -------------------------
for df in [train_data, validation_data]:
    if all(c in df.columns for c in ["nir", "ndmi"]):
        df["nir_x_ndmi"] = df["nir"] * df["ndmi"]
    if all(c in df.columns for c in ["swir16", "swir22"]):
        df["swir_ratio"] = df["swir16"] / (df["swir22"] + 1e-6)
    if all(c in df.columns for c in ["green", "nir"]):
        df["green_nir_ratio"] = df["green"] / (df["nir"] + 1e-6)
    if "ndmi" in df.columns:
        df["ndmi_squared"] = df["ndmi"] ** 2
    if "mndwi" in df.columns:
        df["mndwi_squared"] = df["mndwi"] ** 2
    if "ppt" in df.columns and "ndmi" in df.columns:
        df["ndmi_x_ppt"] = df["ndmi"] * df["ppt"]
    if "ppt" in df.columns and "mndwi" in df.columns:
        df["mndwi_x_ppt"] = df["mndwi"] * df["ppt"]
    if "tmax" in df.columns and "pet" in df.columns:
        df["tmax_x_pet"] = df["tmax"] * df["pet"]
    if "tmin" in df.columns and "pet" in df.columns:
        df["tmin_x_pet"] = df["tmin"] * df["pet"]
    if "tmax" in df.columns and "ppt" in df.columns:
        df["tmax_x_ppt"] = df["tmax"] * df["ppt"]
    if "tmin" in df.columns and "ppt" in df.columns:
        df["tmin_x_ppt"] = df["tmin"] * df["ppt"]
    if "tmax" in df.columns:
        df["tmax_squared"] = df["tmax"] ** 2
    if "tmin" in df.columns:
        df["tmin_squared"] = df["tmin"] ** 2

# -------------------------
# Soil
# -------------------------
soil_cols = [
    "soil_ph", "soil_clay", "soil_sand", "soil_silt",
    "soil_carbon", "soil_bulk_density", "soil_cec"
]

for c in soil_cols:
    if c in train_data.columns and c in validation_data.columns:
        train_data[c] = pd.to_numeric(train_data[c], errors="coerce")
        validation_data[c] = pd.to_numeric(validation_data[c], errors="coerce")

        med = train_data[c].median()

        train_data[f"{c}_missing"] = train_data[c].isna().astype(int)
        validation_data[f"{c}_missing"] = validation_data[c].isna().astype(int)

        train_data[c] = train_data[c].fillna(med)
        validation_data[c] = validation_data[c].fillna(med)

for df in [train_data, validation_data]:
    if "soil_clay" in df.columns and "soil_sand" in df.columns:
        df["soil_texture_ratio"] = df["soil_clay"] / (df["soil_sand"] + 1e-6)
    if "soil_clay" in df.columns and "ppt" in df.columns:
        df["soil_clay_x_ppt"] = df["soil_clay"] * df["ppt"]
    if "soil_sand" in df.columns and "pet" in df.columns:
        df["soil_sand_x_pet"] = df["soil_sand"] * df["pet"]
    if "soil_carbon" in df.columns and "ndmi" in df.columns:
        df["soil_carbon_x_ndmi"] = df["soil_carbon"] * df["ndmi"]
    if "soil_cec" in df.columns and "ppt" in df.columns:
        df["soil_cec_x_ppt"] = df["soil_cec"] * df["ppt"]
    if "soil_ph" in df.columns and "soil_carbon" in df.columns:
        df["soil_ph_x_carbon"] = df["soil_ph"] * df["soil_carbon"]

# -------------------------
# Landcover
# -------------------------
for df in [train_data, validation_data]:
    if "landcover" in df.columns:
        df["landcover"] = pd.to_numeric(df["landcover"], errors="coerce")
        if df["landcover"].notna().any():
            df["landcover"] = df["landcover"].fillna(df["landcover"].mode()[0])

# -------------------------
# Target transform for DRP
# -------------------------
if "dissolved_reactive_phosphorus" in train_data.columns:
    upper_clip = train_data["dissolved_reactive_phosphorus"].quantile(0.99)
    train_data["drp_clipped"] = train_data["dissolved_reactive_phosphorus"].clip(upper=upper_clip)
    train_data["drp_log"] = np.log1p(train_data["drp_clipped"])

print("Training data shape:", train_data.shape)
print("Validation data shape:", validation_data.shape)

print("\nMissing values remaining in train:")
print(train_data.isna().sum().sort_values(ascending=False).head(30))

print("\nMissing values remaining in validation:")
print(validation_data.isna().sum().sort_values(ascending=False).head(30))

feature_fill_cols = [
    "NDMI", "MNDWI", "green", "nir", "swir16", "swir22",
    "soil_clay", "soil_sand", "soil_silt", "soil_carbon",
    "soil_bulk_density", "soil_ph", "soil_cec"
]

for col in feature_fill_cols:
    if col in train_data.columns:
        train_data[f"{col}_missing"] = train_data[col].isna().astype(int)
        median_val = train_data[col].median()
        train_data[col] = train_data[col].fillna(median_val)

        if col in validation_data.columns:
            validation_data[f"{col}_missing"] = validation_data[col].isna().astype(int)
            validation_data[col] = validation_data[col].fillna(median_val)

if "dist_coast_km" in train_data.columns:
    train_data["dist_coast_log"] = np.log1p(train_data["dist_coast_km"])
    validation_data["dist_coast_log"] = np.log1p(validation_data["dist_coast_km"])

# ===== FIX DATE COLUMN AFTER LOWERCASE RENAMING =====

for df in [train_data, validation_data]:
    df["sample_date"] = pd.to_datetime(df["sample_date"], errors="coerce")
    df["sample_year"] = df["sample_date"].dt.year
    df["sample_month"] = df["sample_date"].dt.month
    df["sample_dayofyear"] = df["sample_date"].dt.dayofyear

# Optional: if you don't want raw date in the model later
# keep sample_year / sample_month / sample_dayofyear and drop sample_date before training
print(train_data[["sample_date", "sample_year", "sample_month", "sample_dayofyear"]].head())
print(validation_data[["sample_date", "sample_year", "sample_month", "sample_dayofyear"]].head())

train_data["location_id"] = (
    train_data["latitude"].astype(str) + "_" + train_data["longitude"].astype(str)
)